# Running Simulation Evaluation Locally

The evaluation results reported in the paper are produced on a cloud cluster using Ray (see `vla_foundry/tri/lbm_eval/eval_campaigns/`). This tutorial shows how to run simulation evaluation locally on your own machine using Docker, and serves as a foundation for building an evaluation setup tailored to your own compute resources.

The simulation runs inside a **Docker container**, while the **policy server** runs on the host. The two communicate over **gRPC**. By the end of this notebook you will be able to:
- Run a single-task evaluation manually (policy server + Docker sim)
- Automate multi-task, multi-GPU evaluation with `run_evaluation.py`
- Load, aggregate, and visualize results using the built-in dashboard and plotting utilities

## Overview

| Step | Objective | Key File(s) |
|------|-----------|-------------|
| 0 | Set configuration variables | *(this notebook)* |
| 1 | Download a model checkpoint | `examples/deployment/lbm_eval/download_model_from_wandb.py` |
| 2 | Run a single-task evaluation manually | `vla_foundry/inference/robotics/inference_policy.py`, Docker |
| 3 | Automate evaluation: multi-task, multi-GPU, parameter comparison | `vla_foundry/eval/run_evaluation.py` |
| 4 | View and analyze results | `vla_foundry/eval/data_loading.py`, `vla_foundry/eval/stats.py`, `vla_foundry/eval/results_explorer.py` |
| 5 | Troubleshooting reference | *(this notebook)* |

## Architecture

When getting started, it is easiest to run evaluation in **two terminals**: one for the policy server (host) and one for the simulation (Docker). They communicate over gRPC on `localhost:50051`:

```
Terminal 1 (Host)                Terminal 2 (Docker)
┌──────────────────────┐        ┌──────────────────────┐
│  inference_policy.py  │  gRPC  │  Drake simulation    │
│  (policy server)      │◄──────►│  (lbm_eval Docker)   │
│                       │        │                      │
│  Loads checkpoint,    │        │  Runs evaluation     │
│  generates actions    │        │  episodes, checks    │
│  from observations    │        │  success criteria    │
└──────────────────────┘        └──────────────────────┘
```

### Prerequisites
- NVIDIA GPU
- [NVIDIA Container Toolkit](https://docs.nvidia.com/datacenter/cloud-native/container-toolkit/latest/install-guide.html) installed
- Docker installed
- `uv` installed (see [main README](../README.md#installation))
- The **`Python (vla_foundry)`** Jupyter kernel — install it once from the repo root:
  ```bash
  bash tutorials/install_kernel.sh
  ```
  This syncs dependencies (inference, dashboard, etc.) and registers the kernel.
- Restart your notebook kernel after installing the vla_foundry kernel and select it for execution.
- Sufficient local disk space for rollout data

---

## Step 0: Configuration

Set the variables below once. Every subsequent cell references them, so you only need to change values here.

In [ ]:
import os
import subprocess

# TODO(release): remove AWS_PROFILE once checkpoint is publicly accessible
# Uncomment and set if your AWS credentials require a specific profile:
# os.environ["AWS_PROFILE"] = "your-profile-name"

# Change to the repo root (works regardless of where the notebook is opened from)
repo_root = subprocess.check_output(["git", "rev-parse", "--show-toplevel"], text=True).strip()
os.chdir(repo_root)
print(f"Working directory: {os.getcwd()}")

In [ ]:
# ---- Configuration ----

TASK = "BimanualPutRedBellPepperInBin"  # Task name (PascalCase)
# TODO(release): verify episode indices match the published evaluation protocol
NUM_EPISODES = "0:5"  # Seed range (start:end, exclusive). Paper default: 0:200
OUTPUT_DIR = "rollouts"  # Where results are written
NUM_GPUS = 1  # GPUs for automated evaluation
POLICY_PORT = 50051  # gRPC port for policy server
MAX_SAMPLE_SIZE = 200  # Max episodes per model per task (for sequential statistical test)

DOCKER_IMAGE = "toyotaresearch/lbm-eval-oss:vla-foundry"

# ---- Checkpoint ----
# To evaluate a different checkpoint, change S3_PATH and CHECKPOINT_DIR.
# TODO(release): update S3_PATH to public checkpoint location
S3_PATH = "s3://tri-ml-datasets-uw2/vla_foundry/model_checkpoints/diffusion_policy/ablations/multitask/100m/2026_01_07-23_38_39-model_diffusion_policy-lr_5e-05-bsz_1024_converted"
CHECKPOINT_DIR = "experiments/vla_foundry_checkpoint"

---

## Step 1: Download a Model Checkpoint

Download the pre-trained checkpoint from S3. This fetches only the files needed for inference (config, normalization stats, and model weights).

If you already have a local checkpoint, skip this cell and set `CHECKPOINT_DIR` above to point to it.

In [ ]:
# Download model checkpoint from S3 into CHECKPOINT_DIR.
!uv run python examples/deployment/lbm_eval/download_model_from_wandb.py \
    --s3-path {S3_PATH} \
    --output {CHECKPOINT_DIR}

---

## Step 2: Manual Single-Task Evaluation

This section walks through running a single task end-to-end so you can see how the pieces fit together. Results are written to a separate `rollouts_manual/` directory — Step 3's `run_evaluation.py` writes to `OUTPUT_DIR` with the structured layout expected by the analysis tools in Step 4.

### 2a. Launch the Policy Server

The policy server loads the model checkpoint and waits for gRPC observation requests from the simulator.

| Parameter | Description | Default |
|-----------|-------------|---------|
| `--num_flow_steps` | Diffusion denoising steps | 8 |
| `--open_loop_steps` | Actions executed before re-planning | 8 |
| `--device` | `cuda` or `cpu` | `cuda` |

The next cell launches the server in the background and waits until it is accepting connections before returning.

**Optional:** To visualize observations and actions in real time, uncomment the `VISUALIZER` line below. This requires `uv sync --group visualization`. The server log (`rollouts_manual/.policy_server.log`) will contain a URL like:

```
[rerun_backend] Open in browser: http://localhost:9090/?url=rerun%2Bhttp://127.0.0.1:9876/proxy
```

In [ ]:
import socket
import subprocess
import time
from pathlib import Path

MANUAL_DIR = "rollouts_manual"

# Launch policy server in the background.
# Logs are written to {MANUAL_DIR}/.policy_server.log.
manual_path = Path(MANUAL_DIR)
manual_path.mkdir(parents=True, exist_ok=True)
policy_log = manual_path / ".policy_server.log"
policy_log_fh = policy_log.open("w")  # noqa: SIM115 — must outlive Popen
policy_env = {**os.environ, "CUDA_VISIBLE_DEVICES": "0"}
# Uncomment to enable the rerun visualizer (requires: uv sync --group visualization):
policy_env["VISUALIZER"] = "rerun"

uv_groups = ["--group", "inference"]
if "VISUALIZER" in policy_env:
    uv_groups += ["--group", "visualization"]

policy_proc = subprocess.Popen(
    [
        "uv",
        "run",
        *uv_groups,
        "python",
        "vla_foundry/inference/robotics/inference_policy.py",
        "--checkpoint_directory",
        CHECKPOINT_DIR,
        "--num_flow_steps",
        "8",
        "--open_loop_steps",
        "8",
        "--device",
        "cuda",
    ],
    env=policy_env,
    stdout=policy_log_fh,
    stderr=subprocess.STDOUT,
)

# Wait for the server to accept connections before proceeding (timeout 180s).
# Model loading typically takes 30-60s depending on GPU.
print(f"Policy server launched (PID {policy_proc.pid}), waiting for port {POLICY_PORT}...")
server_ready = False
for i in range(180):
    if policy_proc.poll() is not None:
        break
    try:
        with socket.create_connection(("localhost", POLICY_PORT), timeout=1):
            server_ready = True
            break
    except OSError:
        if i > 0 and i % 10 == 0:
            print(f"  Still waiting... ({i}s elapsed)")
        time.sleep(1)

# Always print the log so far.
policy_log_fh.flush()
print("\n--- Policy server log ---")
print(policy_log.read_text())

if server_ready:
    print(f"Policy server ready on localhost:{POLICY_PORT}")
elif policy_proc.poll() is not None:
    raise RuntimeError(f"Policy server exited with code {policy_proc.returncode}.")
else:
    raise TimeoutError(f"Policy server did not start on port {POLICY_PORT} within 180s.")

### 2b. Pull the Docker Image

The simulation runs inside a Docker container that includes the Drake physics engine and the LBM evaluation harness.

In [ ]:
!docker pull {DOCKER_IMAGE}

### 2c. Run Evaluation Episodes

The Docker container connects to the policy server over gRPC, runs each episode, and writes results to the mounted `rollouts_manual/` directory.

**Key environment variables:**

| Variable | Description | Default |
|----------|-------------|---------|
| `LAUNCH_DEMONSTRATION_INDICES` | Episode seed range (`start:end`, exclusive) | `0:200` |
| `NUM_PROCESSES` | Parallel episodes inside the container | `1` |
| `POLICY_HOST` | gRPC policy server hostname | `localhost` |
| `POLICY_PORT` | gRPC policy server port | `50051` |
| `RECORD_VIDEO` | Save MP4 videos (`1` = yes) | `0` |
| `VIDEO_FPS` | Video frame rate | `10` |

In [ ]:
# Create output directory with open permissions (Docker runs as a different user).
!mkdir -p {MANUAL_DIR} && chmod 777 {MANUAL_DIR}

# Run evaluation episodes inside Docker.
!docker run --rm --network host \
    --runtime=nvidia \
    --gpus all \
    --device /dev/dri \
    --group-add video \
    --group-add $(stat -c '%g' /dev/dri/renderD128) \
    -e NVIDIA_DRIVER_CAPABILITIES=all \
    -e LAUNCH_DEMONSTRATION_INDICES={NUM_EPISODES} \
    -e RECORD_VIDEO=1 \
    -v $(pwd)/{MANUAL_DIR}:/tmp/lbm/rollouts \
    {DOCKER_IMAGE} \
    bash launch_sim.sh {TASK}

### 2d. Cleanup

Stop the background policy server and any running Docker evaluation containers.

In [ ]:
# Stop the policy server.
if policy_proc.poll() is None:
    policy_proc.terminate()
    policy_proc.wait(timeout=10)
    print("Policy server stopped.")
else:
    print(f"Policy server already exited (code {policy_proc.returncode}).")

# Kill any remaining policy servers on the port.
!kill $(lsof -ti :{POLICY_PORT}) 2>/dev/null \
    && echo "Killed process on port {POLICY_PORT}." || true

# Stop any Docker evaluation containers.
!docker kill $(docker ps -q --filter ancestor={DOCKER_IMAGE}) 2>/dev/null \
    && echo "Docker containers stopped." \
    || echo "No running containers found."

---

## Step 3: Automated Multi-Task Evaluation

[`run_evaluation.py`](../vla_foundry/eval/run_evaluation.py) automates Steps 2a–2c above: it launches policy servers (one per GPU), distributes tasks across GPUs with no idle time, and prints progress. When a GPU finishes a task it immediately picks up the next one.

### Setting `--max_sample_size`

Every `run_evaluation.py` invocation requires `--max_sample_size`. This is the **maximum number of episodes per model per task** you plan to ever collect, and it configures the stopping boundary for the [sequential statistical test (STEP)](https://tri-ml.github.io/step/) used to compare models. It must be decided **before seeing any results** to maintain statistical validity — changing it after the fact invalidates the test.

Choose a value based on your compute budget. The paper uses 200 episodes, so `--max_sample_size 200` is a good default. Note that `--num_episodes` can be smaller — you can start with a subset (e.g., `0:50`) to get early results and collect more later up to your budget. See [`STATISTICAL_COMPARISON.md`](STATISTICAL_COMPARISON.md) for details on incremental evaluation and combining results across runs.

### Scaling throughput

The simulation (physics + rendering) is the bottleneck, not the policy server — GPU utilization is typically low. Uncomment the flags below to increase parallelism:

| Flag | Effect |
|------|--------|
| `--num_gpus N` | Use N GPUs, each with its own policy server. |
| `--num_processes N` | Run N parallel episodes *within* each Docker container. Start with 5 and increase while monitoring `nvidia-smi`. |
| `--tasks_per_gpu N` | Run N tasks concurrently on each GPU, each with its own policy server and Docker container. Each policy server uses ~7–10 GB VRAM; with 49 GB GPUs you can fit ~5 concurrent tasks. |

> **Note:** Full evaluation (200 episodes x 22 tasks) takes many hours. If running from a terminal rather than this notebook, use `tmux` or `screen` to prevent SSH disconnects from killing the process.

In [ ]:
# Quick sanity check: 2 episodes, 2 tasks, 1 GPU.
# !uv run python vla_foundry/eval/run_evaluation.py {CHECKPOINT_DIR} \
#     --num_episodes 0:2 --max_sample_size {MAX_SAMPLE_SIZE} \
#     --tasks PutMugOnSaucer TurnCupUpsideDown

# Full evaluation: 200 episodes per task (paper-comparable), all tasks.
# Uncomment --num_gpus, --tasks_per_gpu, --num_processes to scale up.
# !uv run python vla_foundry/eval/run_evaluation.py {CHECKPOINT_DIR} \
#     --num_episodes 0:200 --max_sample_size {MAX_SAMPLE_SIZE} \
#     # --num_gpus 3 \
#     # --tasks_per_gpu 3 \
#     # --num_processes 5

In [ ]:
# Compare different inference parameters using the same checkpoint.
# Results are written to a separate directory to keep them isolated.

COMPARE_DIR = "rollouts_comparison"
COMPARE_TASKS = "BimanualPutRedBellPepperInBin PutMugOnSaucer TurnCupUpsideDown"

# !uv run python vla_foundry/eval/run_evaluation.py {CHECKPOINT_DIR} \
#     --num_episodes 0:10 --max_sample_size {MAX_SAMPLE_SIZE} \
#     --tasks {COMPARE_TASKS} \
#     --model_name flow_steps_4 --num_flow_steps 4 \
#     --output_dir {COMPARE_DIR}

# !uv run python vla_foundry/eval/run_evaluation.py {CHECKPOINT_DIR} \
#     --num_episodes 0:10 --max_sample_size {MAX_SAMPLE_SIZE} \
#     --tasks {COMPARE_TASKS} \
#     --model_name flow_steps_8 --num_flow_steps 8 \
#     --output_dir {COMPARE_DIR}

---

## Step 4: Viewing and Analyzing Results

`load_episodes` scans a results directory and returns a flat list of episode dicts. It expects the structured layout produced by `run_evaluation.py` in Step 3: `{model}/{Task}/rollouts/{timestamp}/results.json`. (Step 2's manual output uses a different layout and is not compatible.)

Three modules work together to turn results into insights:

1. **`data_loading.py`** — scans the filesystem, parses result JSONs, validates and combines runs
2. **`stats.py`** — confidence intervals (Clopper-Pearson), statistical tests (STEP), Plotly charts
3. **`results_explorer.py`** — interactive Gradio dashboard with summary tables, charts, and video playback

Below we use the first two programmatically. The Gradio dashboard is shown at the end.

> **For details on statistical testing methodology** — including how to set experimental budgets, interpret Compact Letter Display (CLD) groups, and combine results across evaluation runs — see [`STATISTICAL_COMPARISON.md`](STATISTICAL_COMPARISON.md).

In [ ]:
from pathlib import Path

from vla_foundry.eval.data_loading import aggregate_episodes, load_episodes

root = Path(OUTPUT_DIR)
episodes, pending_by, crashed_by, max_sample_size = load_episodes(root)

print(f"Loaded {len(episodes)} episodes")
if max_sample_size:
    print(f"Max sample size per model: {max_sample_size}")
if episodes:
    print(f"\nSample episode:\n{episodes[0]}")

In [ ]:
from vla_foundry.eval.stats import clopper_pearson_ci

stats = aggregate_episodes(episodes, ci_fn=clopper_pearson_ci, pending_by=pending_by, crashed_by=crashed_by)

# Print a summary table.
print(f"{'Task':<30} {'Model':<15} {'Success Rate':>13} {'N':>5} {'90% CI':>20}")
print("-" * 88)
for s in sorted(stats, key=lambda x: x["pct"], reverse=True):
    ci = f"[{s['ci_low']:.1%}, {s['ci_high']:.1%}]"
    print(f"{s['task']:<30} {s['model']:<15} {s['pct']:>12.1f}% {s['total']:>5} {ci:>20}")

### Viewing Episode Recordings

Each episode loaded by `load_episodes` includes paths to its video recording (if `RECORD_VIDEO=1` was set) and 3D HTML replay. The cell below displays the first available video inline. You can change the filter to view specific tasks, models, or outcomes.

In [ ]:
from IPython.display import Video, display

# Filter episodes (modify to taste):
#   - All episodes: filtered = episodes
#   - Successes only: filtered = [e for e in episodes if e["success"]]
#   - Failures only: filtered = [e for e in episodes if not e["success"]]
#   - Specific task: filtered = [e for e in episodes if e["task"] == "PutMugOnSaucer"]
filtered = episodes

# Display the first episode with a video recording.
for ep in filtered:
    if ep.get("recording_video"):
        print(f"Task: {ep['task']}  Model: {ep['model']}  Episode: {ep['demo_id']}  Success: {ep['success']}")
        print(f"Video: {ep['recording_video']}")
        display(Video(ep["recording_video"], embed=True, width=640))
        break
else:
    print("No video recordings found. Set RECORD_VIDEO=1 in the Docker command to enable recording.")

### Interactive Gradio Dashboard (optional)

For a full interactive experience — summary table, bar/violin/spider charts, paginated episode video playback, and statistical significance testing — launch the Gradio dashboard:

```bash
uv run --group dashboard python vla_foundry/eval/results_explorer.py rollouts/
```

This starts a server at `http://localhost:8505`. Filter by task or model with the dropdowns, and click **Refresh** to pick up new results while evaluation is running.

> **Browser compatibility:** The dashboard has been tested on Chrome. Video playback in the Episode Recordings tab does not work properly on Safari.

To compare multiple models, run evaluations with different `--model_name` values and point the dashboard at the same output directory.

In [ ]:
# Launch the Gradio dashboard (this blocks -- run in a separate terminal or use &).
# !uv run --group dashboard python vla_foundry/eval/results_explorer.py {OUTPUT_DIR}

---

## Step 5: Troubleshooting

| Issue | Solution |
|-------|----------|
| **Docker container logs** | Simulation output is saved to `{OUTPUT_DIR}/{model}/{Task}/.docker.log`. Check these if a task fails or hangs. |
| **Connection refused / simulation hangs** | Ensure the policy server is fully initialized (`LBMDiffusionPolicy initialized` in logs) before starting Docker. Check `.policy_server_gpu{N}_slot{M}.log` for errors. |
| **GPU rendering errors (EGL)** | Verify NVIDIA Container Toolkit is installed and `/dev/dri` is accessible. |
| **Permission denied on rollouts** | Run `mkdir -p rollouts && chmod 777 rollouts` before starting. |
| **Slow first episode** | Normal — Drake downloads model packages and compiles the scene on the first run. |
| **OOM** | Policy server and simulation share the GPU. Reduce `--tasks_per_gpu` or `--num_processes` to lower concurrent memory usage. |
| **Script errors** | If `run_evaluation.py` fails, check the policy server and Docker logs listed in the output. |
| **Stale containers** | `docker kill $(docker ps -q)` to clean up. |
| **Reproducing paper numbers** | Use 200 episodes (`0:200`). Results may vary across machines due to hardware and environment differences. |
| **Diagnosing crashes** | Check `rollouts/**/results.json` — episodes with `total_time: 0` and a gRPC traceback in `failure_message` are infrastructure crashes, not eval failures. |

---

## Cleanup

Run this cell to kill all policy servers and Docker evaluation containers.

In [ ]:
# Kill all policy servers on the default port.
!kill $(lsof -ti :{POLICY_PORT}) 2>/dev/null \
    && echo "Killed process(es) on port {POLICY_PORT}." \
    || echo "No process on port {POLICY_PORT}."

# Kill all evaluation Docker containers.
!docker kill $(docker ps -q --filter ancestor={DOCKER_IMAGE}) 2>/dev/null \
    && echo "Docker containers stopped." \
    || echo "No running containers found."